In [4]:
import numpy as np, pandas as pd, anndata as ad, scanpy as sc
a = ad.read_h5ad("../data/adamson_2016_upr_perturb_seq.h5ad")
print(a)

AnnData object with n_obs × n_vars = 65337 × 32738
    obs: 'perturbation', 'read count', 'UMI count', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'ncounts', 'ngenes', 'percent_mito', 'percent_ribo', 'nperts'
    var: 'ensembl_id', 'ncounts', 'ncells'
    layers: None (.X)


In [5]:
a.X = a.X.astype("float32")
sc.pp.normalize_total(a, target_sum=1e4)
sc.pp.log1p(a)
print(a.X.max())

7.701263


In [8]:
controls = ["63(mod)_pBA580", "Gal4-4(mod)_pBA582"]
skip = set(controls) | {"missing"}
labels = a.obs["perturbation"].astype(object).fillna("missing").astype(str).values
print("cells with no label:", (labels == "missing").sum())

def mean_expr(mask):
    return np.asarray(a.X[mask].mean(axis=0)).ravel()

control_mean = mean_expr(np.isin(labels, controls))

rows, names = [], []
for p in sorted(set(labels) - skip):
    mask = labels == p
    if mask.sum() >= 20:
        rows.append(mean_expr(mask) - control_mean)
        names.append(p)

Y = pd.DataFrame(np.vstack(rows), index=names, columns=a.var_names)
Y.index = Y.index.str.replace(r"_p.*$", "", regex=True)
Y = Y.groupby(level=0).mean()
Y = Y[Y.index.isin(Y.columns)]
print(Y.shape)

cells with no label: 2613
(88, 32738)


In [9]:
rng = np.random.default_rng(0)
order = rng.permutation(len(Y))
n_test = round(0.2 * len(Y))
Y_test = Y.iloc[order[:n_test]]
Y_train = Y.iloc[order[n_test:]]
print(len(Y_train), "train,", len(Y_test), "test")

70 train, 18 test


In [10]:
def l2_error(pred, truth):
    return np.linalg.norm(pred - truth, axis=1).mean()

pred_none = np.zeros(Y_test.shape)
pred_mean = np.tile(Y_train.mean().values, (len(Y_test), 1))

print("no change:", round(l2_error(pred_none, Y_test.values), 3))
print("mean     :", round(l2_error(pred_mean, Y_test.values), 3))

no change: 3.994
mean     : 3.806


In [11]:
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

K = 10
avg = Y_train.mean().values
centred = Y_train.values - avg

programs = PCA(n_components=K).fit(centred).components_
scores = centred @ programs.T
gene_embed = pd.DataFrame(PCA(n_components=K).fit_transform(centred.T), index=Y.columns)

model = Ridge(alpha=1.0, fit_intercept=False)
model.fit(gene_embed.loc[Y_train.index].values, scores)

pred_linear = model.predict(gene_embed.loc[Y_test.index].values) @ programs + avg
print("linear   :", round(l2_error(pred_linear, Y_test.values), 3))

linear   : 3.767
